# 🧪 4. 평가 하네스 기반 자가 개선 (Self-Correction)

지금까지 만든 에이전트는 "잘 동작하는 것처럼" 보입니다. 그런데 정말 잘 동작하는지, 무엇을 고쳐야 더 좋아지는지는 어떻게 알 수 있을까요?
이번 시간에는 **평가 하네스**로 에이전트를 채점하고, 실패를 분석해 고친 뒤, 다시 채점해 개선과 회귀를 확인하는 **시스템 수준의 자가 개선 루프**를 돌립니다.

---

### 🎯 학습 목표
1. 골든 데이터셋과 **2계층 채점(과정 + 결과)**으로 에이전트를 정량 평가합니다.
2. 점수와 도구 호출 궤적을 보고 **기준 미달 지표와 원인 계층(도구 / 인덱스 / 프롬프트)**을 진단합니다.
3. 원인을 고친 뒤 재평가해 **개선 폭과 회귀 여부**를 확인합니다.

### 🎓 학습 목차

| 파트 | 주제 | 핵심 내용 |
|:---:|:---|:---|
| **Step 0** | 환경 세팅 | |
| **Part 1** | 두 가지 Self-Correction | 런타임 자가 수정 vs 평가 하네스 기반 개선 |
| **Part 2** | 골든 데이터셋 | 실패 유형별 문항 설계, 정답 라벨 |
| **Part 3** | 2계층 채점기 | 과정 40% + 결과 60% (정답 일치 포함), 다중 진단 |
| **Part 4** | 베이스라인 평가 | 점수만 보면 놓치는 것 |
| **Part 5** | 실패 진단 | 궤적 분석 → 원인 계층 찾기 |
| **Part 6** | 개선과 회귀 확인 | 도구 인터페이스 수정 → 재평가 |
| **Part 7** | 운영 관점 | 실행 중 개입(HITL 승인 게이트), 오프라인 평가, CI 품질 게이트 |
| **Part 8** | 정리 | 오늘의 루프 요약, 더 해 보기 |

## 🛠️ Step 0. 환경 세팅

프로젝트 루트 경로를 자동 감지하여 `sys.path`에 등록하고, 환경변수(`.env`)와 비동기 이벤트 루프(`nest_asyncio`)를 설정합니다.

In [ ]:
import os
import sys
import asyncio
import nest_asyncio
from dotenv import load_dotenv

# 1. 프로젝트 루트 경로 자동 설정 (app/ 폴더가 있는 위치까지 상위 탐색)
project_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(project_root, "app")):
        break
    project_root = os.path.dirname(project_root)

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 2. 환경변수 로드 (프로젝트 루트의 .env)
load_dotenv(os.path.join(project_root, ".env"), override=True)

print(f"✅ Working Directory: {os.getcwd()}")

# 3. 주피터 노트북 비동기 루프 중복 방지
nest_asyncio.apply()

# 4. 공통 유틸리티 로드
from app.utils import init_chat_model, normalize_content

from app.eval.rag_eval import JUDGE_MODEL

llm = init_chat_model(model="gemini-3.8-flash", temperature=0.0)           # 평가 대상 에이전트
judge_llm = init_chat_model(model=JUDGE_MODEL, temperature=0.0)             # 채점기: 다른 계열 모델 (gpt-5.4-mini)

---
## 🔁 Part 1. 두 가지 Self-Correction

| 구분 | 런타임 자가 수정 | 평가 하네스 기반 개선 |
|:---|:---|:---|
| **누가 고치나** | 에이전트가 실행 중에 스스로 | 개발자(팀)가 평가 결과를 보고 |
| **무엇을 고치나** | 이번 답변 하나 (재검색, 답변 재작성) | 시스템 자체 (도구, 인덱스, 프롬프트) |
| **비용** | 매 요청마다 지연 시간과 LLM 호출 추가 | 오프라인에서 한 번 |
| **효과의 지속** | 다음 요청에서 같은 실패가 반복될 수 있음 | 고친 내용이 모든 요청에 적용 |

모델의 추론 능력이 좋아지면서 "검색 결과가 부족하면 다시 찾기"는 대부분 ReAct 루프 안에서 모델이 알아서 합니다.
그래서 지금 더 중요한 것은 **에이전트가 스스로 못 고치는 부분, 즉 도구와 데이터 계층의 결함을 찾아내는 장치**입니다.

```
골든 데이터셋 ─▶ 에이전트 실행 ─▶ 2계층 채점 ─▶ 기준 미달 진단 ─▶ 도구/인덱스/프롬프트 수정
      ▲                                                                     │
      └──────────────── 재평가 (개선 확인 + 회귀 확인) ◀──────────────────────┘
```

---
## 📋 Part 2. 골든 데이터셋

좋은 골든셋은 "쉬운 문제 모음"이 아니라 **에이전트가 실패할 수 있는 방식마다 문항을 하나 이상** 둡니다.
각 문항에는 두 가지 **정답 라벨**이 붙어 있어 결과와 과정을 모두 채점할 수 있습니다.

| 라벨 | 채점에 쓰이는 곳 |
|:---|:---|
| **정답 (`ground_truth`)** | 답변의 핵심 사실이 맞는가 (정답 일치), 필요한 근거를 검색했는가 (검색 재현) |
| **써야 하는 도구 (`target_tools`)** | 질문에 맞는 도구를 골랐는가 (도구 선택) |

In [ ]:
import pandas as pd
from IPython.display import display
from app.eval.rag_eval import load_golden

golden = load_golden()
golden_df = pd.DataFrame(golden)[["id", "category", "question", "ground_truth", "target_tools"]]
golden_df.columns = ["문항", "유형", "질문", "정답 (ground_truth)", "써야 하는 도구"]
display(golden_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}).hide(axis="index"))

| 범주 | 확인하려는 실패 |
|:---|:---|
| 단일 규정 / 복수 조항 / 표 데이터 | 검색 품질, 표 수치 왜곡 |
| 약어 / 용어 불일치 | 질문 용어와 문서 용어가 다를 때 |
| 다중 홉 관계 / 이종 소스 결합 | 도구 선택, 앞 결과를 이용한 다음 검색 |
| 동일 소스 다회 검색 | 질의 분할, 불필요한 반복 호출 |
| 존재하지 않는 정보 | 환각 (없는 규정을 지어내는가) |

---
## ⚖️ Part 3. 2계층 채점기

최종 답만 채점하면 **"정답은 맞혔지만 도구를 열 번 넘게 헛돌린"** 에이전트를 잡아내지 못합니다.
그래서 과정과 결과를 따로 채점합니다. (채점기: `app/middleware/rag_eval_harness.py`, 러너: `app/eval/rag_eval.py`)

| 계층 | 지표 (표에 표시되는 이름) | 무엇과 비교하나 | 의미 | 가중치 |
|:---|:---|:---|:---|:-:|
| **과정 (Trajectory, 40%)** | tool_selection_accuracy (도구 선택) | 써야 하는 도구 | 질문에 맞는 도구를 골랐는가 | 1/4 |
| | argument_quality_score | 질문 | 검색어가 핵심을 담았는가 | 1/4 |
| | step_economy_score (호출 효율) | 궤적 | 불필요한 반복 없이 해결했는가 | 1/4 |
| | observation_grounding_score | 궤적 ↔ 답변 | 도구 결과를 답변에 반영했는가 | 1/4 |
| **결과 (Outcome, 60%)** | **answer_correctness (정답 일치)** | **정답** | 답변의 핵심 사실(수치, 조항, 결론)이 정답과 같은가 | **0.4** |
| | faithfulness (근거 충실) | 검색 결과 | 답변이 검색 결과에 근거하는가 (환각 여부) | 0.2 |
| | context_recall (검색 재현) | 정답 ↔ 검색 결과 | 정답에 필요한 사실을 빠짐없이 검색했는가 | 0.2 |
| | answer_relevance | 질문 | 질문에 직접 답하는가 | 0.1 |
| | context_precision | 검색 결과 | 검색 결과 중 쓸모 있는 비율 | 0.1 |

$$\text{종합} = 0.4 \times \text{과정} + 0.6 \times \text{결과}$$

**정답 일치에 가장 큰 가중치를 주는 이유**: 에이전트가 2분기 보고서를 받아 "3분기 증가율은 53.5%"라고 답하면,
그 답은 검색 결과에 충실하고(근거 충실 높음) 질문에도 맞는 형식(관련성 높음)이지만 **틀린 답**입니다. 정답 라벨과 비교해야만 잡아낼 수 있습니다.

각 지표는 LLM이 채점하고, 가중 평균은 **코드가 계산**합니다. LLM에게 평균까지 맡기면 계산이 틀리거나 가중치가 실행마다 달라질 수 있기 때문입니다.

> [!NOTE]
> **채점기는 에이전트와 다른 계열의 모델을 씁니다.** 에이전트는 Gemini, 채점기는 OpenAI `gpt-5.4-mini`입니다.
> 같은 모델이 자기 답을 채점하면 점수가 후해지는 편향(self-preference bias)이 생길 수 있기 때문입니다.
> 채점기는 매 평가마다 문항 수만큼 호출되므로, 상위 모델보다 **충분히 똑똑하면서 저렴한 모델**을 고르는 것이 실무적입니다.

### 진단: 기준에 못 미친 지표를 모두 표시합니다

실패는 대개 **여러 요인이 겹쳐서** 생깁니다. 도구를 잘못 고르면 검색도 부실해지고, 검색이 부실하면 답도 틀립니다.
그래서 실패를 한 가지 유형으로 분류하지 않고 **기준 미달 지표를 모두** 표시합니다. 통과한 문항이라도 기준에 못 미친 지표가 있으면 함께 보여 줍니다.

| 주의할 점 | 이유 |
|:---|:---|
| **"먼저 볼 곳"과 "주요 원인"은 다릅니다** | 진단 항목은 처리 흐름(도구 선택 → 검색 → 생성) 순서로 나열됩니다. 앞단부터 고치면 뒷단 문제가 함께 풀리는 경우가 많아서 **살펴볼 순서**로는 유용하지만, 앞에 있다고 가장 큰 원인이라는 뜻은 아닙니다 |
| **통계는 항목별로 따로 셉니다** | 한 문항에서 첫 번째 진단만 세면, 맨 앞에서 검사하는 항목이 구조적으로 가장 많이 집계되는 착시가 생깁니다. 아래 진단 표는 항목마다 독립적으로 셉니다 |

In [ ]:
from app.eval import rag_eval

rules = pd.DataFrame(rag_eval.DIAGNOSTIC_RULES, columns=["진단", "점수 위치", "지표", "기준 (미만이면 표시)", "의심할 계층"])
display(rules.style.format(precision=1).hide(axis="index"))

---
## 📊 Part 4. 베이스라인 평가

평가 대상은 1교시의 `tool_rag_agent`와 같은 구성(검색 도구 4종 + 같은 시스템 프롬프트)입니다.
한국은행 보고서 도구만 **1교시 초기 버전(v1)**을 노트북에서 직접 정의해 사용합니다.

In [ ]:
from typing import Optional
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver

from app.agents.tool_rag_agent import tools as T
from app.agents.tool_rag_agent.prompt import TOOL_RAG_SYSTEM_PROMPT


@tool("search_bok_reports", parse_docstring=True)
def search_bok_reports_v1(query: str) -> str:
    """한국은행(BOK) 2024년 분기별 주력산업 모니터링 보고서(반도체, 자동차 등)의 실적과 전망을 검색합니다.
    한 번의 호출에는 '특정 1개 분기'와 '특정 1개 산업'만 검색하세요.
    분기 비교나 산업 비교가 필요하면 분기별/산업별로 나누어 여러 번 호출하세요.

    Args:
        query: 검색할 분기별 산업 보고서 관련 자연어 질의 (예: '2024년 3분기 반도체 수출 실적')
    """
    docs = T._vectorstore("bok_industry_reports_store").similarity_search(query, k=2)
    if not docs:
        return f"'{query}' 관련 산업보고서를 찾지 못했습니다."
    return "\n\n".join(f"[{d.metadata.get('source_file', '산업보고서')}]:\n{d.page_content}" for d in docs)


def build_agent(bok_tool):
    tools = [T.search_company_policy, bok_tool, T.search_graph_relations, T.query_enterprise_graphrag]
    return create_agent(model=llm, tools=tools, system_prompt=TOOL_RAG_SYSTEM_PROMPT, checkpointer=InMemorySaver())


agent_v1 = build_agent(search_bok_reports_v1)
print("✅ 베이스라인 에이전트(v1) 준비 완료")

In [ ]:
from app.eval.rag_eval import evaluate_agent, summary_line, results_table, diagnosis_table

print("🧪 베이스라인 평가 (10문항, 동시 4개 실행 / 2~4분 소요)\n")
rows_v1 = await evaluate_agent(agent_v1, golden, judge_llm=judge_llm, concurrency=4)

print("\n📊 " + summary_line(rows_v1))
display(results_table(rows_v1).style.format(precision=2).hide(axis="index"))
print("🩺 진단 항목별 기준 미달 문항 (한 문항이 여러 항목에 걸릴 수 있음)")
display(diagnosis_table(rows_v1).style.hide(axis="index"))

In [ ]:
import matplotlib.pyplot as plt

df_v1 = pd.DataFrame(rows_v1)
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(df_v1["tool_calls"], df_v1["composite"], s=80, color="#3182ce")
for _, r in df_v1.iterrows():
    ax.annotate(r["id"], (r["tool_calls"], r["composite"]), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(0.75, color="#e53e3e", linestyle="--", linewidth=1, label="pass threshold 0.75")
ax.set_xlabel("tool calls")
ax.set_ylabel("composite (overall) score")
ax.set_title("Baseline: score vs. cost")
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

### 💡 점수만 보면 놓치는 것

통과 기준만 보면 대부분 합격일 수 있습니다. 그런데 **아래쪽(점수 낮음)이나 오른쪽(도구 호출 많음)으로 떨어진 점**은
정답을 맞혔더라도 과정이 비효율적이거나 **응답 시간과 비용이 다른 문항의 몇 배**입니다.
(LLM 실행이므로 수치는 실행할 때마다 조금씩 달라집니다.)

---
## 🔍 Part 5. 실패 진단: 궤적에서 원인 계층 찾기

이번 진단 대상은 **`bok-01`(1분기와 3분기 반도체 수출 비교)**입니다. 정답은 맞히지만 같은 소스를 여러 번 검색하는 문항이고,
2교시에서 인덱스에 붙인 **분기 메타데이터**와 직접 연결되어 있어 "인덱스 → 도구 인터페이스"로 이어지는 원인을 따라가기 좋습니다.
(`term-01`, `neg-01`처럼 점수가 낮은 다른 문항은 Part 8의 "더 해 보기"에서 같은 방법으로 진단해 볼 수 있습니다.)
다시 실행해 **어떤 검색어로, 무엇을 받아 왔는지** 궤적을 확인합니다.

In [ ]:
from app.eval.rag_eval import run_case

case = next(c for c in golden if c["id"] == "bok-01")
base = next(r for r in rows_v1 if r["id"] == "bok-01")
print(f"🎯 진단 대상: {case['id']} | {case['question']}")
print(f"   베이스라인: 종합 {base['composite']:.2f} (과정 {base['trajectory']:.2f}, 정답 일치 {base['correctness']:.2f}) | 도구 호출 {base['tool_calls']}회")
print(f"   진단: {', '.join(base['diagnoses']) or '-'}\n")

record = await run_case(agent_v1, case)
for i, step in enumerate(record["trajectory"], 1):
    got = step["response_snippet"].split("]:")[0].lstrip("[")
    print(f"Step {i:>2}: {step['tool_name']}({step['arguments']})")
    print(f"         ↳ 받은 문서: {got}")

### 🧠 무엇이 문제인가?

궤적에서 다음 패턴을 확인해 보세요.
- 에이전트는 도구 설명대로 **분기별로 나눠** 검색합니다. 프롬프트와 도구 설명은 제 역할을 하고 있습니다.
- 그런데 "3분기"를 검색해도 **다른 분기 보고서**가 섞여 돌아오는 경우가 있습니다. 네 분기 보고서의 문장이 거의 같아서, 벡터 유사도만으로는 분기를 확실히 구분하지 못합니다.
- 에이전트는 원하는 분기의 수치를 찾을 때까지 **검색어를 바꿔 가며 반복**합니다.

| 원인 후보 | 판단 |
|:---|:---|
| 프롬프트 / 도구 설명 | 에이전트는 지침대로 분기별로 검색함 → 원인 아님 |
| 인덱스 | 2교시 문맥 헤더에 문서명(분기)이 들어 있어 어느 정도는 구분됨. 하지만 유사도 기반(Soft)이라 **보장되지 않음**. 청크마다 `quarter` 메타데이터도 이미 있음 (아래 확인) |
| **도구 인터페이스** | **분기를 지정할 방법이 없음 → 메타데이터가 있어도 쓸 수 없음** ✅ |

에이전트가 스스로 고칠 수 없는 결함입니다. 아무리 다시 검색해도 도구에 분기 필터(Hard)가 없으니까요.

In [ ]:
import chromadb
from collections import Counter

col = chromadb.PersistentClient(path="app/database/chroma_db").get_collection("bok_industry_reports_store")
metas = col.get(include=["metadatas"])["metadatas"]
print("청크별 quarter 메타데이터 분포:", dict(Counter(m.get("quarter") for m in metas)))

---
## 🛠️ Part 6. 개선과 회귀 확인

인덱스에 이미 있는 `quarter` 메타데이터를 **도구 인자로 열어 줍니다.** 반환 형식에도 분기와 위치를 넣어, 에이전트가 받은 결과의 출처를 바로 알 수 있게 합니다.

In [ ]:
@tool("search_bok_reports", parse_docstring=True)
def search_bok_reports_v2(query: str, quarter: Optional[int] = None) -> str:
    """한국은행(BOK) 2024년 분기별 주력산업 모니터링 보고서(반도체, 자동차 등)의 실적과 전망을 검색합니다.
    한 번의 호출에는 '특정 1개 분기'와 '특정 1개 산업'만 검색하세요.
    분기 비교나 산업 비교가 필요하면 분기별/산업별로 나누어 여러 번 호출하세요.
    특정 분기의 수치가 필요하면 quarter를 지정해야 다른 분기 보고서가 섞이지 않습니다.

    Args:
        query: 검색할 산업과 지표 (예: '반도체 수출 증가율')
        quarter: 2024년 분기 번호(1~4). 지정하면 해당 분기 보고서에서만 검색합니다.
    """
    search_kwargs = {"filter": {"quarter": quarter}} if quarter in (1, 2, 3, 4) else {}
    docs = T._vectorstore("bok_industry_reports_store").similarity_search(query, k=3, **search_kwargs)
    if not docs:
        return f"'{query}' 관련 산업보고서를 찾지 못했습니다."
    return "\n\n".join(
        f"[{d.metadata.get('breadcrumb', '산업보고서')}]:\n"
        f"{d.metadata.get('raw_content', d.page_content)}"
        for d in docs
    )


agent_v2 = build_agent(search_bok_reports_v2)
print("✅ 개선 에이전트(v2) 준비 완료: search_bok_reports에 quarter 필터 추가")

### 6.1 진단 대상 문항 재실행

In [ ]:
record_v2 = await run_case(agent_v2, case)
for i, step in enumerate(record_v2["trajectory"], 1):
    got = step["response_snippet"].split("]:")[0].lstrip("[")
    print(f"Step {i:>2}: {step['tool_name']}({step['arguments']})")
    print(f"         ↳ 받은 문서: {got}")
print(f"\n⏱️ {record_v2['duration_s']}s | 도구 호출 {len(record_v2['trajectory'])}회 (베이스라인 {base['tool_calls']}회)")

### 6.2 전체 재평가: 개선 + 회귀 확인

한 문항이 좋아졌다고 끝이 아닙니다. 도구 설명이나 반환 형식을 바꾸면 **다른 문항의 행동이 달라질 수 있으므로** 전체를 다시 채점합니다.

In [ ]:
print("🧪 v2 전체 재평가\n")
rows_v2 = await evaluate_agent(agent_v2, golden, judge_llm=judge_llm, concurrency=4)

from app.eval.rag_eval import compare_table

print("\n📊 v1 " + summary_line(rows_v1))
print("📊 v2 " + summary_line(rows_v2))
display(compare_table(rows_v1, rows_v2).style.format(precision=2).hide(axis="index"))
print("🩺 v2 진단 항목별 기준 미달 문항")
display(diagnosis_table(rows_v2).style.hide(axis="index"))

### 💡 이번 개선에서 배울 점

- 고친 것은 **프롬프트도 모델도 아닌 도구 인터페이스**입니다. 2교시 인덱싱에서 만든 메타데이터가 도구 인자로 열리면서 비로소 쓸모가 생겼습니다.
- `delta`가 ±0.05 안쪽인 문항은 LLM 실행과 채점의 흔들림 범위로 봅니다.
- **개선이든 회귀든, 표시가 붙은 문항은 먼저 "이번 변경과 관련된 도구를 썼는가?"를 확인하세요.**
  `search_bok_reports`를 쓰지 않는 문항(사내 규정, 없는 정보 등)의 점수가 크게 오르거나 내렸다면, 변경 때문이 아니라 실행마다 생기는 변동일 가능성이 큽니다.
  같은 문항을 2~3번 다시 돌려 평균을 보고, 그래도 차이가 나면 궤적을 열어 실제로 행동이 달라졌는지 확인합니다.
  변경과 무관한 개선을 이번 수정의 성과로 착각하는 것도 회귀를 놓치는 것만큼 흔한 실수입니다.
- 이번 결과에서도 남은 약점이 보입니다. `neg-01`(없는 규정)과 `term-01`(용어 불일치)은 정답을 내지만 **검색을 여러 번 반복**합니다. Part 8의 "더 해 보기"에서 같은 루프로 고쳐 볼 수 있습니다.
- 이 루프를 문항마다 반복하는 것이 **시스템 수준의 Self-Correction**입니다.

---
## 🏭 Part 7. 운영 관점: 무엇을 실시간으로, 무엇을 오프라인으로

### 7.1 실행 중 개입: 자동 가드레일과 사람(HITL)

평가 하네스는 실행이 **끝난 뒤** 시스템을 고칩니다. 반대로 실행 **도중**에 개입하는 장치도 있습니다.

| 방식 | 하는 일 | 언제 쓰나 |
|:---|:---|:---|
| **자동 가드레일** | 검색 결과가 부실하면 재검색, 답변의 근거를 검사해 차단 | 재검색은 ReAct 루프가 이미 하므로 대부분 불필요. 매 요청 지연이 늘어나므로, 평가로 **그 실패가 실제로 자주 난다**는 것을 확인한 뒤 고위험 도메인에만 |
| **사람의 개입 (Human-in-the-Loop)** | 에이전트를 멈추고 사람이 승인·수정·거절 | 에이전트가 스스로 판단해 **비싸거나 위험한 일**을 하려 할 때 |

단순 검색-답변 RAG에서는 HITL의 비중이 크지 않습니다. 읽기만 하므로 잘못돼도 틀린 답변 하나로 끝나기 때문입니다.
**RAG가 에이전트가 되면서** 에이전트가 스스로 도구를 고르고 반복 호출하므로, 사람이 개입할 지점이 생깁니다.

| 개입 지점 | 왜 필요한가 | 이 과정의 예 |
|:---|:---|:---|
| **① 비싸거나 위험한 도구 호출 전 승인** | 자율성을 주면 비용과 위험을 통제할 장치가 필요 | GraphRAG Global (LLM 다수 호출), Text2SQL의 실제 DB 실행, 민감 문서 접근 |
| **② 답변 확정 전 검토** | 틀린 답의 대가가 큰 도메인 | 근거 점수가 낮은 답변을 담당자 검토 대기열로 |
| **③ 질문 명확화** | 추측하며 검색을 반복하는 것보다 되묻는 것이 싸고 정확 | `term-01` "과장급": 규정에 없는 직급 → 반복 검색 대신 확인 질문 |
| **④ 피드백 → 평가 루프** | 사람의 판정이 골든셋으로 쌓여야 평가가 계속 자람 | 7.3의 "운영 중 실패 → 골든셋 추가" |

### 7.2 실습: 비싼 도구 앞에 승인 게이트 두기

LangChain의 `HumanInTheLoopMiddleware`를 `create_agent`에 붙이면, 지정한 도구를 호출하기 **직전에 실행이 멈춥니다(interrupt)**.
사람이 결정을 내려 주면 **체크포인터에 저장된 상태에서 그대로 재개(resume)**합니다. 1교시에 붙인 체크포인터가 여기서 필요한 이유입니다.

| 도구 | 정책 (`interrupt_on`) | 이유 |
|:---|:---|:---|
| `search_company_policy`, `search_bok_reports`, `search_graph_relations` | 자동 실행 | 빠르고 싼 읽기 전용 검색 |
| `query_enterprise_graphrag` | **승인 필요** (`approve` / `edit` / `reject`) | Global 검색은 커뮤니티 리포트 전체를 Map-Reduce로 종합해 LLM 호출이 많음 |

```mermaid
flowchart LR
    A["🤖 에이전트<br/>도구 호출 제안"] --> G{"🛑 HITL 정책"}
    G -->|"검색 도구"| R["⚡ 즉시 실행"]
    G -->|"GraphRAG"| I["⏸️ interrupt<br/>체크포인터에 상태 저장"]
    I --> H{"👤 사람"}
    H -->|"approve"| R
    H -->|"edit: 인자 수정"| R
    H -->|"reject + 사유"| F["❌ 실행 안 함<br/>사유를 에이전트에 전달"]
    R --> A
    F --> A
```

In [ ]:
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.types import Command

hitl_agent = create_agent(
    model=llm,
    tools=[T.search_company_policy, search_bok_reports_v2, T.search_graph_relations, T.query_enterprise_graphrag],
    system_prompt=TOOL_RAG_SYSTEM_PROMPT,
    checkpointer=InMemorySaver(),   # 멈춘 상태를 저장해 두어야 재개할 수 있음
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "query_enterprise_graphrag": {"allowed_decisions": ["approve", "edit", "reject"]},
            },
            description_prefix="🛑 GraphRAG 호출은 비용이 큰 작업이라 승인이 필요합니다.",
        )
    ],
)


def show_pending(result) -> bool:
    """멈춘 실행이 있으면 승인 대기 중인 도구 호출을 보여 줍니다."""
    if "__interrupt__" not in result:
        return False
    request = result["__interrupt__"][0].value
    for action, review in zip(request["action_requests"], request["review_configs"]):
        print(f"⏸️ 승인 대기: {action['name']}({action.get('args', action.get('arguments'))})")
        print(f"   허용된 결정: {review['allowed_decisions']}")
    return True


def resume(agent, result, decision: dict, config: dict):
    """대기 중인 도구 호출마다 같은 결정을 내려 재개합니다. (병렬 호출이면 승인 요청도 여러 건)"""
    pending = result["__interrupt__"][0].value["action_requests"]
    return agent.invoke(Command(resume={"decisions": [decision] * len(pending)}), config=config)


def show_answer(result):
    used = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    print(f"🔧 사용한 도구: {used}")
    print("💬 최종 답변:\n" + normalize_content(result["messages"][-1].content)[:700])


print("✅ HITL 에이전트 준비 완료: query_enterprise_graphrag 호출 전 승인 필요")

**시나리오 A: 승인 (approve)**

전사 차원의 질문을 하면 에이전트는 GraphRAG Global 검색을 고릅니다. 실행은 도구 호출 직전에 멈춥니다.

In [ ]:
config_a = {"configurable": {"thread_id": "hitl-approve"}}
question_a = "전사 프로젝트들의 공통 리스크를 요약해줘."

result = hitl_agent.invoke({"messages": [{"role": "user", "content": question_a}]}, config=config_a)
paused = show_pending(result)
print("\n📌 실행이 멈췄는가?", paused)

In [ ]:
# 사람이 내용을 확인하고 승인하면, 저장된 상태에서 이어서 실행됩니다.
while "__interrupt__" in result:
    result = resume(hitl_agent, result, {"type": "approve"}, config_a)

show_answer(result)

**시나리오 B: 거절 (reject) + 사유 전달**

이번에는 비용을 이유로 거절하고, **대신 무엇을 하라고** 사유에 적어 보냅니다. 에이전트는 사유를 읽고 다른 도구로 답을 찾습니다.

In [ ]:
config_b = {"configurable": {"thread_id": "hitl-reject"}}
question_b = "클라우드운영팀 김철수 수석의 소속팀을 알려줘."

result = hitl_agent.invoke({"messages": [{"role": "user", "content": question_b}]}, config=config_b)

reject_reason = (
    "GraphRAG는 비용이 커서 이 질문에는 승인할 수 없습니다. "
    "특정 인물의 관계 질문이니 search_graph_relations로 조직 그래프를 탐색해서 답하세요."
)
rejected = 0
while show_pending(result):
    print(f"🙅 거절: {reject_reason}\n")
    result = resume(hitl_agent, result, {"type": "reject", "message": reject_reason}, config_b)
    rejected += 1

if rejected == 0:
    print("ℹ️ 이번 실행에서는 에이전트가 GraphRAG 없이 답해 승인 요청이 없었습니다. (다시 실행하면 달라질 수 있습니다)\n")
show_answer(result)

💡 정리하면 이렇습니다.
- 승인 게이트는 **도구 단위 정책**이라, 에이전트 코드나 프롬프트를 고치지 않고 붙였다 뗄 수 있습니다.
- 거절할 때 **사유와 대안**을 함께 주면, 에이전트는 멈추지 않고 다른 경로로 답을 찾습니다.
- 실제 서비스에서는 이 interrupt를 Chainlit/웹 UI의 **승인 버튼**과 연결합니다. 사람이 버튼을 누를 때까지 상태는 체크포인터(SQLite 등)에 저장되어 있습니다.
- 어떤 도구에 게이트를 둘지는 감으로 정하지 않습니다. 평가 리포트의 **도구별 호출 수와 소요 시간**이 근거가 됩니다.

### 7.3 평가 하네스의 운영 배치

```
사용자 ─▶ RAG Agent ─▶ 즉시 응답 (지연 0)
              │
              └─(비동기)─▶ 로그 큐 ─▶ 5~10% 샘플링 ─▶ 2계층 채점 ─▶ 대시보드 (LangSmith 등)

코드/프롬프트 변경 ─▶ CI ─▶ 골든셋 전체 채점 ─▶ 평균 ≥ 기준 & 회귀 없음 ─▶ 배포 승인
```

| 단계 | 방식 | 목적 |
|:---|:---|:---|
| 개발 중 | 이 노트북, `python -m app.eval.rag_eval` | 실패 진단과 개선 확인 |
| 배포 전 | CI 품질 게이트 (골든셋 전체) | 회귀 차단 |
| 운영 중 | 트래픽 샘플 비동기 채점 | 골든셋에 없던 실패 발견 → 골든셋에 추가 |

운영 중 발견한 실패 사례를 골든셋에 추가하면 루프가 닫힙니다. 골든셋은 한 번 만들고 끝나는 것이 아니라 **계속 자라는 회귀 테스트**입니다.

---
## 🏁 Part 8. 정리

| 단계 | 오늘 한 일 | 현업에서는 |
|:---|:---|:---|
| **평가** | 골든셋 10문항, 과정(Trajectory) + 결과(Outcome) 2계층 채점 | 실패 유형마다 문항을 두고, 운영 중 실패 사례로 계속 키움 |
| **진단** | 점수와 도구 호출 궤적으로 원인 계층(프롬프트 / 인덱스 / 도구) 찾기 | 점수만이 아니라 **호출 수와 소요 시간**을 함께 봄 |
| **수정** | 인덱스의 `quarter` 메타데이터를 **도구 인자로** 열기 | 에이전트가 스스로 못 고치는 결함은 도구와 데이터 계층에 있음 |
| **재평가** | 전체 골든셋으로 개선과 회귀 확인 | CI 품질 게이트로 회귀를 배포 전에 차단 |
| **실행 중 개입** | 비싼 도구 앞에 HITL 승인 게이트 | 비용·위험이 큰 도구만 골라서, 평가 데이터를 근거로 |